In [ ]:
import pandas as pd
import numpy as np

# Load the raw canvas grades csv file
df = pd.read_csv('/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/NEUROSCI380L/2026-03-30T1203_Grades-NEUROSCI_380L.01L.Sp26.csv')

# output path to save the modified csv file to
output_path = '/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/NEUROSCI380L/Final_Modified_Grades_NEUROSCI_380L_01L_Sp26_with_Averages.csv'

In [ ]:
# Convert all score columns to float, handling non-numeric entries
for col in df.columns[4:]:  # Adjust this as necessary
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Extract points possible from the first row after the headers and convert them to float
points_possible = df.iloc[0, 4:].astype(float)  # Adjust index if your data starts elsewhere

# Convert scores to percentages, starting from the second row (index 1)
df.iloc[1:, 4:] = df.iloc[1:, 4:].div(points_possible)

# Drop the 'Points Possible' row if it's no longer needed
df = df.drop(index=0)

# Define the categories and their weights, allowing for partial matches in the column names
categories = {
    'TRA': {'weight': 0.20, 'exclude': ['Final Exam: TRA', 'Exam', 'myths and mysteries']},
    'IRA': {'weight': 0.35, 'exclude': ['Final Exam: IRA', 'Exam', 'myths and mysteries']},
    'Clinical case set': {'weight': 0.20, 'exclude': ['Clinical case set (self-directed learning seminar) (322896)']},
    'Self-Directed Learning Seminar Written Critique Upload': {'weight': 0.05, 'exclude': None},
    'TRA: Final Exam': {'weight': 0.10, 'exclude': ['TRA: Final exam 2025']},
    'IRA: Final Exam': {'weight': 0.10, 'exclude': None}
}

# Initialize a column for the final average
df['Final Average'] = 0

# To store used columns for each category
used_columns = {}

# Loop through each category to calculate weighted averages
for category, info in categories.items():
    
    # Find columns that contain the category substring and do not include any excluded substrings
    columns = [col for col in df.columns if category.lower() in col.lower() and
               all(excl.lower() not in col.lower() for excl in (info['exclude'] if info['exclude'] else [])) and
               'score' not in col.lower() and 'grade' not in col.lower()]
    if not columns:
        print(f'No columns found for category: {category}')
        continue
    
    if category == 'IRA':
        # df[columns] is shape (n_students, n_ira_assignments)
        # For each row, sort scores ascending and drop the 2 lowest
        ira_scores = df[columns].values  # numpy array
        ira_sorted = np.sort(ira_scores, axis=1)  # NaN goes to the end
        ira_dropped = ira_sorted[:, 2:]  # drop the 2 lowest per row
        df[category + ' Average'] = np.nanmean(ira_dropped, axis=1)
    else:
        df[category + ' Average'] = df[columns].mean(axis=1)

    df['Final Average'] += df[category + ' Average'] * info['weight']
        
    used_columns[category] = columns  # Store used columns for later use
    
    print(category, used_columns[category])

# Save the modified DataFrame to a new CSV file
df.to_csv(output_path, index=False)

# Print only the columns that exist in the DataFrame
columns_to_print = ['Student', 'Final Average'] + [col for col in df.columns if 'Average' in col and col != 'Final Average']
print(f"DataFrame saved to {output_path}")
print(df[columns_to_print])

# Print used columns for each specific category
for category in ['TRA', 'IRA', 'Clinical case set']:
    print(f"Columns used for {category} calculations: {used_columns[category]}")


TRA ['TRA: Blood supply to the brain and spinal cord (322899)', 'TRA: Surface anatomy of the forebrain (322910)', 'TRA: Internal anatomy of the forebrain  (322904)', 'TRA: Surface anatomy of the brainstem and spinal cord  (322909)', 'TRA: Hypothalamus and visceral motor system (322902)', 'TRA: Internal anatomy of the spinal cord and brainstem (322905)', 'TRA: Somatic sensory pathways  (322908)', 'TRA: Visual pathways  (322912)', 'TRA: Upper and Lower Motor Neurons  (322911)', 'TRA: Basal ganglia  (322898)']
IRA ['IRA: Internal anatomy of the forebrain (322863)', 'IRA: Internal anatomy of the forebrain_makeup (340406)', 'IRA: Blood supply to the brain and spinal cord (322850)', 'IRA: Blood supply to the brain and spinal cord_makeup (340405)', 'IRA: Surface anatomy of the forebrain (322886)', 'IRA: Surface anatomy of the forebrain makeup (341886)', 'IRA: Surface anatomy of the forebrain___makeup (337346)', 'IRA: Surface anatomy of the forebrain____makeup (340407)', 'IRA: Surface anatomy 

/var/folders/f0/1x9ky0yd3nj4p68sphk83ngr0000gn/T/ipykernel_27090/2542778525.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Final Average'] = 0
/var/folders/f0/1x9ky0yd3nj4p68sphk83ngr0000gn/T/ipykernel_27090/2542778525.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[category + ' Average'] = df[columns].mean(axis=1)
/var/folders/f0/1x9ky0yd3nj4p68sphk83ngr0000gn/T/ipykernel_27090/2542778525.py:53: RuntimeWarning: Mean of empty slice
  df[category + ' Average'] = np.nanmean(ira_dropped, axis=1)
/var/folders/f0/1x